# Objetivo específico 2 — Ingeniería de datos (ETL)

**Versión preparada para GitHub y Google Colab.** Las tres fuentes se descargan directamente del repositorio, sin subir archivos durante cada ejecución.

**Antes de ejecutar:** publicar en `Objetivo_2_Diseno_ETL/datos/` estos archivos de entrada, con los nombres exactos:
- `DATOS_HISTORICOS.csv`: reportes históricos y nuevos ya consolidados.
- `CATALOGO_PRODUCTOS.csv`: catálogo maestro de productos.
- `SUCURSALES.csv`: catálogo de sucursales.

No publiques información interna de CBN sin autorización. Los archivos públicos deben estar autorizados o adecuadamente anonimizados; quitar nombres no garantiza anonimato. El cuaderno **no incluye** los registros originales, claves privadas ni salidas con datos.

Ejecuta **Entorno de ejecución → Ejecutar todas** cuando las tres fuentes estén publicadas. Los CSV de salida se generan en el entorno de Colab.


In [ ]:
# ETL — extracción desde GitHub, sin carga manual de archivos
import pandas as pd
from io import BytesIO
from urllib.request import urlopen
from urllib.parse import quote
from urllib.error import HTTPError, URLError

BASE_URL = (
    "https://raw.githubusercontent.com/adrianamarcelayavi-cmyk/"
    "inteligencia-negocios-series-temporales-supermercados/main/Objetivo_2_Diseno_ETL/datos"
)

def leer_csv_github(nombre):
    url = f"{BASE_URL}/{quote(nombre)}"
    try:
        with urlopen(url, timeout=30) as respuesta:
            contenido = respuesta.read()
    except (HTTPError, URLError) as error:
        raise RuntimeError(
            f"No se pudo obtener {nombre} desde GitHub. "
            "Verifica el nombre, la ruta, la rama main y que el archivo sea público."
        ) from error
    for encoding in ("utf-8-sig", "utf-8", "latin1"):
        try:
            return pd.read_csv(
                BytesIO(contenido), sep=None, engine="python",
                encoding=encoding, dtype=str
            )
        except UnicodeDecodeError:
            pass
    raise ValueError(f"No se logró decodificar {nombre}")

# Fuentes de entrada: se publican una sola vez en GitHub
df_original = leer_csv_github("DATOS_HISTORICOS.csv")
df_catalogo = leer_csv_github("CATALOGO_PRODUCTOS.csv")
df_sucursales = leer_csv_github("SUCURSALES.csv")

# Normalización de los tipos que necesita el cuaderno original
# Los identificadores permanecen como texto para preservar ceros iniciales.
if "CANTIDAD" in df_original.columns:
    df_original["CANTIDAD"] = pd.to_numeric(df_original["CANTIDAD"], errors="coerce")

print("BASE CONSOLIDADA:", df_original.shape)
print("CATÁLOGO DE PRODUCTOS:", df_catalogo.shape)
print("SUCURSALES:", df_sucursales.shape)
print("\nCOLUMNAS DE CADA TABLA:")
print("Históricos:", df_original.columns.tolist())
print("Catálogo:", df_catalogo.columns.tolist())
print("Sucursales:", df_sucursales.columns.tolist())


In [ ]:
# ETL - TRANSFORMACIÓN DE DATOS
# PASO 2: Selección de columnas

columnas_principales = [
    "FECHA",
    "TIPO_REPORTE",
    "SUCURSAL",
    "CODIGO_BARRAS",
    "CODIGO_PRODUCTO",
    "PRODUCTO",
    "CANTIDAD",
    "FECHA_VENCIMIENTO"
]

# Trabajar con una copia de los datos originales
df = df_original[columnas_principales].copy()

# Verificar el resultado
print("DIMENSIONES DESPUÉS DE LA LIMPIEZA:", df.shape)

print("\nCOLUMNAS CONSERVADAS:")
print(df.columns.tolist())

display(df.head())


TIPO DE DATOS

In [ ]:
# ETL - TRANSFORMACIÓN DE DATOS
# PASO 3: Revisión de tipos de datos

print("INFORMACIÓN GENERAL DE LA BASE:")
df.info()

print("\nTIPOS DE DATOS POR COLUMNA:")
print(df.dtypes)

print("\nVALORES FALTANTES POR COLUMNA:")
print(df.isna().sum())

print("\nDIMENSIONES DE LA BASE:")
print(df.shape)


In [ ]:
# ETL - TRANSFORMACIÓN DE DATOS
# PASO 4: Conversión de tipos de datos

# Fechas
for columna in ["FECHA", "FECHA_VENCIMIENTO"]:
    df[columna] = pd.to_datetime(
        df[columna],
        format="%d/%m/%Y",
        errors="coerce"
    )

# Variables categóricas
for columna in ["TIPO_REPORTE", "SUCURSAL"]:
    df[columna] = df[columna].astype("category")

# Identificadores
for columna in ["CODIGO_BARRAS", "CODIGO_PRODUCTO"]:
    df[columna] = df[columna].astype("string")

# Nombre completo del producto
df["PRODUCTO"] = df["PRODUCTO"].astype("string")

# Verificar conversiones
print("TIPOS DE DATOS:")
print(df.dtypes)

print("\nCATEGORÍAS DE TIPO_REPORTE:")
print(df["TIPO_REPORTE"].cat.categories.tolist())

print("\nCATEGORÍAS DE SUCURSAL:")
print(df["SUCURSAL"].cat.categories.tolist())

print("\nDIMENSIONES:")
print(df.shape)

In [ ]:
# PASO 5: Validación de fechas y cantidades

print("VALORES FALTANTES:")
print(df.isna().sum())

print("\nFECHAS DE VENCIMIENTO FALTANTES POR REPORTE:")
print(
    df.groupby("TIPO_REPORTE", observed=True)
      ["FECHA_VENCIMIENTO"]
      .apply(lambda x: x.isna().sum())
)

print("\nCANTIDADES NULAS O NO POSITIVAS:")
print("Cantidades nulas:", df["CANTIDAD"].isna().sum())
print("Cantidades menores o iguales a cero:",
      (df["CANTIDAD"] <= 0).sum())

print("\nRANGO DE FECHAS DE REGISTRO:")
print("Fecha mínima:", df["FECHA"].min())
print("Fecha máxima:", df["FECHA"].max())

print("\nREGISTROS CONSERVADOS:", len(df))


REGISTROS DUPLICADOS

In [ ]:
# PASO 6: Identificación de duplicados

# Detectar duplicados exactos
duplicados = df.duplicated(keep=False)

print("REGISTROS DUPLICADOS ADICIONALES:")
print(df.duplicated().sum())

print("\nFILAS INVOLUCRADAS EN DUPLICADOS:")
print(duplicados.sum())

# Mostrar los registros para revisión
if duplicados.any():
    display(df.loc[duplicados].sort_values(
        by=["FECHA", "SUCURSAL", "PRODUCTO"]
    ))
else:
    print("No se encontraron duplicados exactos.")

print("\nREGISTROS ACTUALES:", len(df))


In [ ]:
# PASO 7: Eliminación de duplicados exactos

# Guardar una copia antes de eliminar duplicados
df_antes_duplicados = df.copy()

# Cantidad inicial de registros
registros_antes = len(df)

# Eliminar duplicados exactos, conservando el primero
df = df.drop_duplicates(keep="first").reset_index(drop=True)

# Verificación
print("REGISTROS ANTES:", registros_antes)
print("DUPLICADOS ELIMINADOS:", registros_antes - len(df))
print("REGISTROS DESPUÉS:", len(df))
print("DUPLICADOS RESTANTES:", df.duplicated().sum())
print("DIMENSIONES FINALES:", df.shape)


VALORES NULOS

In [ ]:
# PASO 8: Validación de valores nulos

print("VALORES NULOS POR COLUMNA:")
print(df.isna().sum())

print("\nPORCENTAJE DE VALORES NULOS:")
print((df.isna().mean() * 100).round(2))

print("\nFECHAS FALTANTES SEGÚN TIPO DE REPORTE:")
print(
    df.groupby("TIPO_REPORTE", observed=True)
      ["FECHA_VENCIMIENTO"]
      .apply(lambda x: x.isna().sum())
)

print("\nDIMENSIONES ACTUALES:")
print(df.shape)


ESTANDARIZACIÓN DE TEXTOS

In [ ]:
# PASO 9: Limpieza y estandarización de textos

columnas_texto = [
    "TIPO_REPORTE",
    "SUCURSAL",
    "PRODUCTO"
]

# Conservar los valores anteriores para comparar
textos_antes = df[columnas_texto].copy()

# Convertir temporalmente a texto y normalizar
for columna in columnas_texto:
    df[columna] = (
        df[columna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )

# Restaurar tipo categórico donde corresponde
df["TIPO_REPORTE"] = df["TIPO_REPORTE"].astype("category")
df["SUCURSAL"] = df["SUCURSAL"].astype("category")

# Contar valores modificados por columna
print("VALORES MODIFICADOS:")
for columna in columnas_texto:
    cambios = (
        textos_antes[columna].astype("string")
        != df[columna].astype("string")
    ).fillna(False).sum()

    print(f"{columna}: {cambios}")

print("\nTIPOS DE REPORTE:")
print(df["TIPO_REPORTE"].cat.categories.tolist())

print("\nSUCURSALES:")
print(df["SUCURSAL"].cat.categories.tolist())

print("\nDIMENSIONES:")
print(df.shape)


ESTANDARIZAR MEDIANTE CATÁLOGO MAESTRO


In [ ]:
# PASO 10: Resolver claves duplicadas del catálogo maestro
maestro_limpio = df_catalogo.copy()
claves = ["CODIGO_BARRAS", "CODIGO_PRODUCTO"]
for columna in claves:
    maestro_limpio[columna] = maestro_limpio[columna].astype("string").str.strip()

# Identificar diferencias de denominación para la misma combinación de códigos.
conflictos = maestro_limpio[maestro_limpio.duplicated(subset=claves, keep=False)]
print("FILAS CON CLAVES REPETIDAS EN EL CATÁLOGO:", len(conflictos))
if not conflictos.empty:
    print("ATENCIÓN: se conservará la primera aparición de cada clave. "
          "Revisa que sea la denominación autorizada antes de utilizar la base.")
# La clave debe ser única para la validación many_to_one posterior.
maestro_limpio = maestro_limpio.drop_duplicates(subset=claves, keep="first").copy()
print("REGISTROS DEL CATÁLOGO:", len(maestro_limpio))
print("PARES DUPLICADOS RESTANTES:", maestro_limpio.duplicated(subset=claves).sum())


COINCIDENCIA DE PARES DE CÓDIGOS

In [ ]:
# PASO 11: Validación con el catálogo limpio

claves = ["CODIGO_BARRAS", "CODIGO_PRODUCTO"]

# Crear copias para la comparación
base_validacion = df.copy()
catalogo_validacion = maestro_limpio.copy()

# Uniformar los códigos como texto
for columna in claves:
    base_validacion[columna] = (
        base_validacion[columna].astype("string").str.strip()
    )
    catalogo_validacion[columna] = (
        catalogo_validacion[columna].astype("string").str.strip()
    )

# Comparar ambos códigos
validacion_productos = base_validacion.merge(
    catalogo_validacion[claves + ["PRODUCTO"]].rename(
        columns={"PRODUCTO": "PRODUCTO_MAESTRO"}
    ),
    on=claves,
    how="left",
    validate="many_to_one",
    indicator=True
)

# Mostrar resultados
print("REGISTROS ORIGINALES:", len(df))
print("REGISTROS DESPUÉS DE LA COMPARACIÓN:",
      len(validacion_productos))

print("\nRESULTADO DE LA VALIDACIÓN:")
print(validacion_productos["_merge"].value_counts())

print("\nREGISTROS SIN CORRESPONDENCIA:")
print((validacion_productos["_merge"] == "left_only").sum())

print("\nDIMENSIONES:")
print(validacion_productos.shape)


REVISIÓN DE REGISTROS SIN COINCIDENCIA

In [ ]:
# PASO 12: Revisar registros sin correspondencia

sin_correspondencia = validacion_productos[
    validacion_productos["_merge"] == "left_only"
].copy()

print("REGISTROS SIN CORRESPONDENCIA:")
print(len(sin_correspondencia))

display(
    sin_correspondencia[
        [
            "FECHA",
            "SUCURSAL",
            "CODIGO_BARRAS",
            "CODIGO_PRODUCTO",
            "PRODUCTO",
            "PRODUCTO_MAESTRO"
        ]
    ]
)

print("\nCONTROL DE REGISTROS:")
print("Base original importada:", len(df_original))
print("Base transformada:", len(df))
print("Diferencia:", len(df_original) - len(df))


In [ ]:
# PASO 13: Exclusión de productos eventuales

# Conservar por separado los registros excluidos
df_eventuales = validacion_productos[
    validacion_productos["_merge"] == "left_only"
].copy()

# Conservar los registros validados con el catálogo
df_validado = validacion_productos[
    validacion_productos["_merge"] == "both"
].copy()

# Eliminar únicamente la columna auxiliar de validación
df_validado = df_validado.drop(columns=["_merge"])

# Verificar resultados
print("REGISTROS ANTES:", len(validacion_productos))
print("PRODUCTOS EVENTUALES EXCLUIDOS:", len(df_eventuales))
print("REGISTROS CONSERVADOS:", len(df_validado))

print("\nCONTROL DE CONSERVACIÓN:")
print(len(df_eventuales) + len(df_validado))

display(
    df_eventuales[
        ["CODIGO_BARRAS", "CODIGO_PRODUCTO", "PRODUCTO"]
    ].drop_duplicates()
)


ESTANDARIZACIÓN DE NOMBRES

In [ ]:
# ETL - TRANSFORMACIÓN DE DATOS
# PASO 14: Estandarización de nombres de productos

# Crear una copia de los registros validados
df_limpio = df_validado.copy()

# Conservar el nombre registrado originalmente
df_limpio["PRODUCTO_ORIGINAL"] = df_limpio["PRODUCTO"].copy()

# Asignar el nombre del catálogo maestro
df_limpio["PRODUCTO"] = (
    df_limpio["PRODUCTO_MAESTRO"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Eliminar la columna auxiliar
df_limpio = df_limpio.drop(columns=["PRODUCTO_MAESTRO"])

# Verificar resultados
print("REGISTROS CONSERVADOS:", len(df_limpio))

print("\nPRODUCTOS SIN NOMBRE:")
print(df_limpio["PRODUCTO"].isna().sum())

print("\nNOMBRES DISTINTOS ESTANDARIZADOS:")
print(df_limpio["PRODUCTO"].nunique())

print("\nCOMPARACIÓN DE NOMBRES:")
display(
    df_limpio[
        ["CODIGO_BARRAS",
         "CODIGO_PRODUCTO",
         "PRODUCTO_ORIGINAL",
         "PRODUCTO"]
    ].head(15)
)


In [ ]:
# PASO 15: Verificación de QUILMES y QUILLMES

# Buscar ambas variantes en la base estandarizada
revision_quilmes = df_limpio[
    df_limpio["PRODUCTO"].str.contains(
        r"QUILMES|QUILLMES",
        case=False,
        na=False
    )
]

print("NOMBRES EN LA BASE ESTANDARIZADA:")
print(revision_quilmes["PRODUCTO"].value_counts())

print("\nREGISTROS CON QUILLMES (DOBLE L):")
errores = df_limpio[
    df_limpio["PRODUCTO"].str.contains(
        "QUILLMES", case=False, na=False
    )
]
print(len(errores))

print("\nCÓDIGOS Y NOMBRES IDENTIFICADOS:")
display(
    revision_quilmes[
        ["CODIGO_BARRAS", "CODIGO_PRODUCTO", "PRODUCTO"]
    ].drop_duplicates()
)


ESTANDARIZACIÓN DE SUCURSALES

In [ ]:
# PASO 16: Verificación de sucursales

print("COLUMNAS DEL CATÁLOGO DE SUCURSALES:")
print(df_sucursales.columns.tolist())

print("\nCATÁLOGO DE SUCURSALES:")
display(df_sucursales)

print("\nSUCURSALES EN LA BASE TRANSFORMADA:")
print(df_limpio["SUCURSAL"].value_counts())

print("\nCANTIDAD DE SUCURSALES DISTINTAS:")
print(df_limpio["SUCURSAL"].nunique())


In [ ]:
# PASO 17: Validación de sucursales

# Obtener nombres del catálogo
sucursales_maestras = set(
    df_sucursales["SUCURSAL"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Obtener nombres de la base transformada
sucursales_base = set(
    df_limpio["SUCURSAL"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identificar sucursales sin correspondencia
sin_correspondencia = (
    sucursales_base - sucursales_maestras
)

print("SUCURSALES EN EL CATÁLOGO:",
      len(sucursales_maestras))

print("SUCURSALES EN LA BASE:",
      len(sucursales_base))

print("SUCURSALES SIN CORRESPONDENCIA:",
      len(sin_correspondencia))

print("\nDETALLE:")
print(sorted(sin_correspondencia))

print("\nREGISTROS CONSERVADOS:",
      len(df_limpio))


VALIDACIÓN DE COHERENCIA DE FECHAS

In [ ]:
# PASO 18: Validación de coherencia de fechas

# Crear copia para revisión
revision_fechas = df_limpio.copy()

# Identificar inconsistencias
vencidos_incorrectos = (
    (revision_fechas["TIPO_REPORTE"] == "VENCIDO") &
    (
        revision_fechas["FECHA_VENCIMIENTO"].isna() |
        (revision_fechas["FECHA_VENCIMIENTO"] >
         revision_fechas["FECHA"])
    )
)

proximos_incorrectos = (
    (revision_fechas["TIPO_REPORTE"] == "PROXIMO") &
    (
        revision_fechas["FECHA_VENCIMIENTO"].isna() |
        (revision_fechas["FECHA_VENCIMIENTO"] <=
         revision_fechas["FECHA"])
    )
)

# Unir las inconsistencias identificadas
inconsistencias = revision_fechas[
    vencidos_incorrectos | proximos_incorrectos
]

print("VENCIDOS CON FECHAS INCONSISTENTES:",
      vencidos_incorrectos.sum())

print("PRÓXIMOS CON FECHAS INCONSISTENTES:",
      proximos_incorrectos.sum())

print("TOTAL DE REGISTROS A REVISAR:",
      len(inconsistencias))

display(
    inconsistencias[
        ["FECHA", "TIPO_REPORTE", "SUCURSAL",
         "PRODUCTO", "FECHA_VENCIMIENTO"]
    ].head(20)
)


In [ ]:
# PASO 19: Clasificación de inconsistencias de fechas

import numpy as np

# Copia para revisión
df_revision = df_limpio.copy()

# Identificar inconsistencias
condiciones = [
    (df_revision["TIPO_REPORTE"] == "VENCIDO") &
    (df_revision["FECHA_VENCIMIENTO"] > df_revision["FECHA"]),

    (df_revision["TIPO_REPORTE"] == "PROXIMO") &
    (df_revision["FECHA_VENCIMIENTO"] <= df_revision["FECHA"])
]

descripciones = [
    "VENCIDO CON FECHA FUTURA",
    "PROXIMO CON FECHA VENCIDA"
]

# Agregar estado de validación
df_revision["ESTADO_FECHA"] = np.select(
    condiciones,
    descripciones,
    default="CORRECTO SEGUN REGLA"
)

print("RESUMEN DE VALIDACIÓN:")
print(df_revision["ESTADO_FECHA"].value_counts())

print("\nREGISTROS PARA REVISIÓN:")
display(
    df_revision[
        df_revision["ESTADO_FECHA"] != "CORRECTO SEGUN REGLA"
    ][
        ["FECHA", "TIPO_REPORTE", "SUCURSAL",
         "PRODUCTO", "FECHA_VENCIMIENTO",
         "ESTADO_FECHA"]
    ]
)


In [ ]:
# PASO 20: Separación de registros inconsistentes

# Registros que cumplen las reglas
df_correctos = df_revision[
    df_revision["ESTADO_FECHA"] == "CORRECTO SEGUN REGLA"
].copy()

# Registros que requieren corrección
df_por_corregir = df_revision[
    df_revision["ESTADO_FECHA"] != "CORRECTO SEGUN REGLA"
].copy()

# Verificación
print("REGISTROS CORRECTOS:", len(df_correctos))
print("REGISTROS POR CORREGIR:", len(df_por_corregir))

print("TOTAL CONTROLADO:",
      len(df_correctos) + len(df_por_corregir))

display(
    df_por_corregir[
        ["FECHA", "TIPO_REPORTE", "SUCURSAL",
         "PRODUCTO", "FECHA_VENCIMIENTO",
         "ESTADO_FECHA"]
    ]
)


In [ ]:
# PASO 21: Preparar tabla de correcciones

columnas_revision = [
    "FECHA",
    "TIPO_REPORTE",
    "SUCURSAL",
    "CODIGO_BARRAS",
    "CODIGO_PRODUCTO",
    "PRODUCTO",
    "FECHA_VENCIMIENTO",
    "ESTADO_FECHA"
]

tabla_correcciones = df_por_corregir[
    columnas_revision
].copy()

# Conservar el índice para identificar cada registro
tabla_correcciones.insert(
    0, "ID_REGISTRO", tabla_correcciones.index
)

# Campos para documentar las correcciones
tabla_correcciones["CAMPO_A_CORREGIR"] = ""
tabla_correcciones["VALOR_CORRECTO"] = ""
tabla_correcciones["OBSERVACION"] = ""

print("REGISTROS PARA REVISIÓN:", len(tabla_correcciones))

display(tabla_correcciones)


In [ ]:
# PASO 22: Propuesta de reclasificación lógica

# Trabajar sobre una copia
df_propuesta = df_limpio.copy()

# Conservar la clasificación original
df_propuesta["TIPO_REPORTE_ORIGINAL"] = (
    df_propuesta["TIPO_REPORTE"].astype("string")
)

# Crear clasificación propuesta
df_propuesta["TIPO_REPORTE_PROPUESTO"] = (
    df_propuesta["TIPO_REPORTE"].astype("string")
)

# Solo revisar VENCIDO y PROXIMO
revisables = df_propuesta["TIPO_REPORTE"].isin(
    ["VENCIDO", "PROXIMO"]
)

fecha_valida = df_propuesta["FECHA_VENCIMIENTO"].notna()

# Reclasificar según la relación entre fechas
vencidos = (
    revisables & fecha_valida &
    (df_propuesta["FECHA_VENCIMIENTO"] <= df_propuesta["FECHA"])
)

proximos = (
    revisables & fecha_valida &
    (df_propuesta["FECHA_VENCIMIENTO"] > df_propuesta["FECHA"])
)

df_propuesta.loc[
    vencidos, "TIPO_REPORTE_PROPUESTO"
] = "VENCIDO"

df_propuesta.loc[
    proximos, "TIPO_REPORTE_PROPUESTO"
] = "PROXIMO"

# Marcar fecha antigua sospechosa
df_propuesta["REVISAR_FECHA"] = (
    df_propuesta["FECHA_VENCIMIENTO"].dt.year.lt(2020)
    .fillna(False)
)

# Mostrar solamente cambios propuestos
cambios = df_propuesta[
    df_propuesta["TIPO_REPORTE_ORIGINAL"] !=
    df_propuesta["TIPO_REPORTE_PROPUESTO"]
]

print("CAMBIOS DE CLASIFICACIÓN PROPUESTOS:", len(cambios))
print("FECHAS ANTIGUAS PARA REVISIÓN:",
      df_propuesta["REVISAR_FECHA"].sum())

display(cambios[[
    "FECHA", "PRODUCTO", "TIPO_REPORTE_ORIGINAL",
    "TIPO_REPORTE_PROPUESTO", "FECHA_VENCIMIENTO",
    "REVISAR_FECHA"
]])


In [ ]:
# PASO 23: Aplicar reclasificación lógica

# Crear copia de trabajo
df_corregido = df_propuesta.copy()

# Identificar cambios que no tienen fecha sospechosa
cambios_aplicables = (
    (
        df_corregido["TIPO_REPORTE_ORIGINAL"] !=
        df_corregido["TIPO_REPORTE_PROPUESTO"]
    )
    & (~df_corregido["REVISAR_FECHA"])
)

# Aplicar cambios de clasificación
df_corregido["TIPO_REPORTE"] = (
    df_corregido["TIPO_REPORTE"].astype("string")
)

df_corregido.loc[
    cambios_aplicables, "TIPO_REPORTE"
] = df_corregido.loc[
    cambios_aplicables, "TIPO_REPORTE_PROPUESTO"
]

# Restaurar tipo categórico
df_corregido["TIPO_REPORTE"] = (
    df_corregido["TIPO_REPORTE"].astype("category")
)

print("RECLASIFICACIONES APLICADAS:",
      cambios_aplicables.sum())

print("REGISTROS PENDIENTES POR FECHA SOSPECHOSA:",
      df_corregido["REVISAR_FECHA"].sum())

print("REGISTROS CONSERVADOS:", len(df_corregido))


In [ ]:
# PASO 24: Verificación de las reclasificaciones

# VENCIDO: vencimiento anterior o igual al reporte
error_vencido = (
    (df_corregido["TIPO_REPORTE"] == "VENCIDO") &
    (
        df_corregido["FECHA_VENCIMIENTO"].isna() |
        (df_corregido["FECHA_VENCIMIENTO"] > df_corregido["FECHA"])
    )
)

# PROXIMO: vencimiento posterior al reporte
error_proximo = (
    (df_corregido["TIPO_REPORTE"] == "PROXIMO") &
    (
        df_corregido["FECHA_VENCIMIENTO"].isna() |
        (df_corregido["FECHA_VENCIMIENTO"] <= df_corregido["FECHA"])
    )
)

print("INCONSISTENCIAS TEMPORALES RESTANTES:",
      (error_vencido | error_proximo).sum())

print("FECHAS SOSPECHOSAS PENDIENTES:",
      df_corregido["REVISAR_FECHA"].sum())

print("REGISTROS CONSERVADOS:", len(df_corregido))

print("\nREGISTRO PENDIENTE:")
display(
    df_corregido.loc[
        df_corregido["REVISAR_FECHA"],
        ["FECHA", "TIPO_REPORTE", "SUCURSAL",
         "PRODUCTO", "FECHA_VENCIMIENTO"]
    ]
)


In [ ]:
# PASO 25: Eliminación del registro incorrecto

# Conservar copia para trazabilidad
df_registro_eliminado = df_corregido[
    df_corregido["REVISAR_FECHA"]
].copy()

# Eliminar de la base de análisis
df_final = df_corregido[
    ~df_corregido["REVISAR_FECHA"]
].copy()

# Reiniciar índices
df_final = df_final.reset_index(drop=True)

# Verificar resultados
print("REGISTROS ANTES:", len(df_corregido))
print("REGISTROS ELIMINADOS:", len(df_registro_eliminado))
print("REGISTROS FINALES:", len(df_final))

print("\nFECHAS SOSPECHOSAS RESTANTES:")
print(df_final["REVISAR_FECHA"].sum())

In [ ]:
# ETL - TRANSFORMACIÓN DE DATOS
# PASO 26: Control final de calidad

print("DIMENSIONES DE LA BASE:")
print(df_final.shape)

print("\nVALORES NULOS:")
print(df_final.isna().sum())

print("\nDUPLICADOS EXACTOS:")
print(df_final.duplicated(
    subset=[
        "FECHA", "TIPO_REPORTE", "SUCURSAL",
        "CODIGO_BARRAS", "CODIGO_PRODUCTO",
        "PRODUCTO", "CANTIDAD", "FECHA_VENCIMIENTO"
    ]
).sum())

print("\nCANTIDADES NO POSITIVAS:")
print((df_final["CANTIDAD"] <= 0).sum())

print("\nREGISTROS POR TIPO DE REPORTE:")
print(df_final["TIPO_REPORTE"].value_counts())

# Verificar coherencia temporal
error_vencido = (
    (df_final["TIPO_REPORTE"] == "VENCIDO") &
    (df_final["FECHA_VENCIMIENTO"] > df_final["FECHA"])
)

error_proximo = (
    (df_final["TIPO_REPORTE"] == "PROXIMO") &
    (df_final["FECHA_VENCIMIENTO"] <= df_final["FECHA"])
)

print("\nINCONSISTENCIAS DE FECHAS:")
print((error_vencido | error_proximo).sum())

print("\nPRODUCTOS DISTINTOS:")
print(df_final["PRODUCTO"].nunique())

print("\nSUCURSALES DISTINTAS:")
print(df_final["SUCURSAL"].nunique())


In [ ]:
# PASO 27: Preparar la base analítica

columnas_finales = [
    "FECHA",
    "TIPO_REPORTE",
    "SUCURSAL",
    "CODIGO_BARRAS",
    "CODIGO_PRODUCTO",
    "PRODUCTO",
    "CANTIDAD",
    "FECHA_VENCIMIENTO"
]

# Conservar una copia con las columnas auxiliares
df_trazabilidad = df_final.copy()

# Crear la base destinada al análisis
df_analitico = df_final[columnas_finales].copy()

print("BASE ANALÍTICA:", df_analitico.shape)
print("BASE DE TRAZABILIDAD:", df_trazabilidad.shape)

print("\nCOLUMNAS DE LA BASE ANALÍTICA:")
print(df_analitico.columns.tolist())

display(df_analitico.head())


In [ ]:
# PASO 28: Control de procedencia de la base

print("REGISTROS DE LA FUENTE ACTUAL:")
print(len(df_original))

print("\nREGISTROS DESPUÉS DE LA TRANSFORMACIÓN:")
print(len(df_analitico))

print("\nREGISTROS EXCLUIDOS POR PRODUCTOS EVENTUALES:")
print(len(df_eventuales))

print("\nREGISTROS EXCLUIDOS POR FECHA INCORRECTA:")
print(len(df_registro_eliminado))

print("\nCONTROL:")
print(
    "Total contabilizado:",
    len(df_analitico)
    + len(df_eventuales)
    + len(df_registro_eliminado)
)


In [ ]:
# PASO 29: Exportación de datos transformados

# Guardar base analítica
df_analitico.to_csv(
    "datos_transformados.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardar trazabilidad de las transformaciones
df_trazabilidad.to_csv(
    "trazabilidad_transformaciones.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos generados correctamente.")
print("Registros analíticos:", len(df_analitico))
print("Registros de trazabilidad:", len(df_trazabilidad))


In [ ]:
# PASO 30: Verificación de los archivos transformados

import pandas as pd

# Leer el archivo generado por el ETL
df_verificacion = pd.read_csv(
    "datos_transformados.csv",
    dtype={
        "CODIGO_BARRAS": "string",
        "CODIGO_PRODUCTO": "string"
    }
)

print("DIMENSIONES:", df_verificacion.shape)

print("\nCOLUMNAS:")
print(df_verificacion.columns.tolist())

print("\nREGISTROS POR TIPO:")
print(df_verificacion["TIPO_REPORTE"].value_counts())

print("\nDUPLICADOS EXACTOS:")
print(df_verificacion.duplicated().sum())


In [ ]:
# PASO 31: Comprobación final del ETL

print("BASE ORIGINAL:", df_original.shape)
print("BASE TRANSFORMADA:", df_analitico.shape)

print("\nREGISTROS POR TIPO:")
print(df_analitico["TIPO_REPORTE"].value_counts())

print("\nDUPLICADOS RESTANTES:")
print(df_analitico.duplicated().sum())


In [ ]:
# PASO 32: Verificación final de tipos de datos

print("TIPOS DE DATOS DE LA BASE ANALÍTICA:")
print(df_analitico.dtypes)

print("\nVALORES NULOS POR COLUMNA:")
print(df_analitico.isna().sum())

print("\nRESUMEN FINAL:")
df_analitico.info()


In [ ]:
# PASO 33: Exportación complementaria para análisis exploratorio
df_analitico.to_csv(
    "datos_limpios_EDA.csv", index=False, encoding="utf-8-sig"
)
print("Archivo datos_limpios_EDA.csv generado en el entorno de Colab.")
# Descarga opcional: desde el panel Archivos de Google Colab.
